###### Content under Creative Commons Attribution license CC-BY 4.0, code under BSD 3-Clause License © 2022  by D. Koehn, T. Meier and J. Stampa, notebook style sheet by L.A. Barba, N.C. Clementi

# Digital Signal Processing: Convolution and Filtering

## Chapter 7 Exercises

### Exercise 7.1

Show that $(\delta \ast f)(\tau) = f(\tau)$. Interpret the result.

**Solution**

According to the lecture, the convolution of two continuous functions is defined as 

\begin{equation}
(\delta \ast f)(\tau)=\int_{-\infty}^{\infty}\delta(t')f(\tau-t')dt'\notag
\end{equation}

The [sifting property of the Dirac delta function](https://en.wikipedia.org/wiki/Dirac_delta_function#Translation) implies that the integral yields the function value $f(\tau-t')$ at the point $t'=0$. Thus, it follows that

\begin{equation}
(\delta \ast f)(\tau)=\int_{-\infty}^{\infty}\delta(t')f(\tau-t')dt'=f(\tau)\notag
\end{equation}

which was to be shown.

### Exercise 7.3

Define a time series of length 200 s that takes the value 1 at 100 s and is otherwise zero. The sampling interval is 0.1 s. Filter the time series using a third-order single-sided Butterworth filter with the following characteristics:
- Low-pass, cutoff frequency 0.02 Hz. 
- High-pass, cutoff frequency 0.02 Hz.

Calculate the amplitude spectrum of the filtered time series. Plot the results and interpret them.

### Exercise 7.4

Calculate a Butterworth bandpass filter with cutoff frequencies of 0.02 Hz and 0.1 Hz. Apply the filter to the sequence of values from the previous exercise on both sides. Interpret the results.

**Solution**

We'll solve Problems 7.3 and 7.4 using Python code. As always, we'll start by importing the usual Python libraries. From the `SciPy` library, we'll need some code to define and apply Butterworth filters ...

In [ ]:
# Import Python libraries
# -----------------------
#%matplotlib inline
from ipywidgets import interactive
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

I want to program this problem in an as modular way as possible so that the code can also be reused in other programs if needed. The `butt_filt` function defines the Butterworth filter and applies it to the time series $x(t)$. 

The parameters **ffiltmin** and **fflitmax** specify the lower and upper cutoff frequencies for a bandpass filter. In the case of a low-pass or high-pass filter, ffiltmin is used as the cutoff frequency. The **order** parameter is used to define the order of the Butterworth filter.

The `SciPy` function [signal.butter](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.butter.html) is used to define the Butterworth filter, which is then applied to the time series using [signal.sosfilt](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.sosfilt.html) is applied to the time series on one side. If the filter is to be applied on both sides, simply use the [signal.sosfiltfilt](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.sosfiltfilt.html) function. The **low**, **high**, and **band** switches activate a low-pass, high-pass, or band-pass Butterworth filter, respectively. The **two_sided** option enables two-sided filtering; by default, one-sided filtering is applied.  

In [ ]:
def butt_filt(x,dt,ffiltmin,ffiltmax,order,low,high,band,two_sided):
    
    fs = 1./dt # sample frequency
    
    # design low-pass filter
    # ----------------------
    if(low==True):
        
        sos = signal.butter(order, ffiltmin, 'lp', fs=fs, output='sos')
            
    # design high-pass filter
    # -----------------------
    if(high==True):
        
        sos = signal.butter(order, ffiltmin, 'hp', fs=fs, output='sos')
    
    # design band-pass filter
    # -----------------------
    if(band==True):
        
        sos = signal.butter(order, [ffiltmin,ffiltmax], 'bp', fs=fs, output='sos')    
    
    # apply filter to time series x(t) either one-sided or two-sided    
    # --------------------------------------------------------------
    if(two_sided==False):
        
        x = signal.sosfilt(sos, x)     # apply one-sided filter
        
    else:
        
        x = signal.sosfiltfilt(sos, x) # apply two-sided filter        
    
    return x

To define the time series as flexibly as possible, we will also write a function called `time_series` for this purpose. First, we will implement the time series as specified in Exercise 7.3 ...

In [ ]:
def time_series():
    
    # Define parameters
    dt = .1                # time sampling [s]
    L = 200.               # length of the time series [s]
    tspike = 100.          # position of spike in time series [s]
    
    # Define time series ...
    t = np.arange(0,L+dt,dt) # compute time vector
    nt = len(t)              # number of samples in time series
    x = np.zeros(nt)         # intialize time series with zeros
    
    # Compute time sample of spike
    ntspike = (int)(tspike // dt)
    
    # Add spike to time series
    x[ntspike] = 1.
    
    return t, x

To calculate the amplitude spectra, we use the `x_fft` function to compute the FFT and frequencies. We can simply copy these from the notebook for [Exercise 3](https://nbviewer.org/github/daniel-koehn/Geophysikalische-Signalverarbeitung/blob/master/exercises/Ex03_FFT.ipynb) ...

In [ ]:
def x_fft(x,dt):
    
    # Fourier transform
    X = np.fft.fft(x)
    
    # Normalize by length of the time series
    N = len(x)
    X = X / (N*dt)
    
    # estimate frequencies    
    freq = np.fft.fftfreq(N, d=dt)
    
    return freq,X

Finally, we implement the `butterworth` function, which defines the time series, designs the filters, applies them to the time series $x(t)$, calculates the amplitude spectra, and visualizes them ...

In [ ]:
def butterworth(low,high,band,two_sided,order):
    
    # Define filter parameters
    ffiltmin = 0.02        # minimum cut-off frequency for Butterworth filter [Hz]
    ffiltmax = 0.1         # maximum cut-off frequency for Butterworth filter [Hz]
    
    # Define time series
    # ------------------
    t, x = time_series()
    nt = len(t)     # number of samples in time series
    dt = t[1]-t[0]  # sample interval [s]
    

    # Compute frequencies and spectrum of time series x(t)
    freq, X = x_fft(x,dt)
    
    # Define and apply Butterworth ...
    # --------------------------------
    if(low==True): # ... low-pass filter
        
        x_low = butt_filt(x,dt,ffiltmin,ffiltmax,order,True,False,False,two_sided)
                
        # fft of low-pass filtered time series
        freq, X_low = x_fft(x_low,dt)
        
    if(high==True): # ... high-pass filter

        x_high = butt_filt(x,dt,ffiltmin,ffiltmax,order,False,True,False,two_sided)
        
        # fft of high-pass filtered time series
        freq, X_high = x_fft(x_high,dt)
        
    if(band==True): # ... band-pass filter
        
        x_band = butt_filt(x,dt,ffiltmin,ffiltmax,order,False,False,True,two_sided)        
        
        # fft of band-pass filtered time series
        freq, X_band = x_fft(x_band,dt)    
    
    # Plot time series x(t)
    # ---------------------
    plt.figure(figsize=(20,10))
    
    plt.subplot(211)
    
    if(low==False and high==False and band==False):
        plt.plot(t, x, 'b-',lw=3,label='x(t)')
        
    if(low==True):
        plt.plot(t, x_low, 'r-',lw=3,label='low-pass x(t)')        
        
    if(high==True):
        plt.plot(t, x_high, 'g-',lw=3,label='high-pass x(t)')
        
    if(band==True):
        plt.plot(t, x_band, 'y-',lw=3,label='band-pass x(t)')    
        
    plt.title('x(t)')
    plt.xlabel('time [s]')
    plt.ylabel('(Filtered) time series x(t)')
    plt.legend()
    plt.grid()
    
    plt.subplot(212)
    
    # plot KKF
    plt.plot(freq[:nt//2], np.abs(X[:nt//2]), 'b-',lw=3,label='X(f)')
    if(low==True):
        plt.plot(freq[:nt//2], np.abs(X_low[:nt//2]), 'r-',lw=3,label='low-pass X(f)')
        
    if(high==True):
        plt.plot(freq[:nt//2], np.abs(X_high[:nt//2]), 'g-',lw=3,label='high-pass X(f)')
        
    if(band==True):
        plt.plot(freq[:nt//2], np.abs(X_band[:nt//2]), 'y-',lw=3,label='band-pass X(f)')
        
    plt.title('Amplitude spectra of (filtered) time series')
    plt.xlabel(r'frequency f [Hz]')
    plt.ylabel('Amplitude')
    plt.legend()
    plt.xlim(0,.2)
    plt.grid()

To make it easier to compare the effects of different filters on the time series, we use interactive Jupyter widgets. In addition to toggles for low-pass, high-pass, and bandpass Butterworth filters, two-sided filtering can also be enabled. The filter order can be interactively adjusted between 1 and 10.

In [ ]:
interactive_plot = interactive(butterworth, low=False, high=False, band=False, two_sided=False, order=(1,10,1))
output = interactive_plot.children[-1]
output.layout.height = '600px'
interactive_plot

The code demonstrates ...

- Applying the filters results in the expected filtering effect on the white spectrum of a time series using the Dirac delta impulse, as determined by the cutoff frequencies. 

- By varying the filter order, the filter’s slope can be adjusted. A steeper slope in the frequency domain results in more pronounced side lobes in the time domain. Accordingly, the filter order should be chosen to achieve a good compromise between the filtering effect and side lobes.

- The one-sided filter results in minimum-phase filtering, whereas the two-sided filter results in zero-phase filtering of the delta impulse. Depending on how the data is to be further processed, analyzed, and interpreted, the choice between a one-sided and a two-sided filter can be critical.

### Exercise 7.5

Modify the set of values so that all values are 1. Repeat the one-sided and two-sided filtering. Interpret the results.

Since the code was written in a modular fashion, we only need to modify the time series in the `time_series` function according to Exercise 7.5 ...

In [ ]:
def time_series():
    
    # Define parameters
    dt = .1                # time sampling [s]
    L = 200.               # length of the time series [s]
    tspike = 100.          # position of spike in time series [s]
    
    # Define time series ...
    t = np.arange(0,L+dt,dt) # compute time vector
    nt = len(t)              # number of samples in time series
    x = np.ones(nt)         # intialize time series with ones
    
    return t, x

... and restart the interactive IPython widget ...

In [ ]:
interactive_plot = interactive(butterworth, low=False, high=False, band=False, two_sided=False, order=(1,10,1))
output = interactive_plot.children[-1]
output.layout.height = '600px'
interactive_plot

If the time series has a constant value (bias), a Dirac delta impulse appears in the amplitude spectrum at a frequency of 0 Hz. Accordingly, ...

- ... the low-pass filter should not alter the time series. This works very well with the two-sided filter. When varying the order of the Butterworth filter, only occasional, somewhat strange axis scalings occur, which attempt to reflect small deviations in the filtered time series. However, due to its minimal-phase property, the one-sided filter has the problem that the filtered time series initially assumes a value of zero and only reaches a value of 1 as time increases.

- ... the high-pass and band-pass filters remove the bias at the specified cutoff frequencies. As with the low-pass filter, the two-sided filter delivers the desired result. The one-sided filter, on the other hand, produces values that deviate significantly from zero at the beginning of the time series.